In [ ]:
# ==============================================================================
# LABORATÓRIO ECONOMÉTRICO: TESTES DE CAUSALIDADE E DINÂMICA COMPARATIVA (VIX II vs EPU)
# ==============================================================================

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

warnings.filterwarnings('ignore')

plt.style.use(
    'seaborn-v0_8-whitegrid'
    if 'seaborn-v0_8-whitegrid' in plt.style.available
    else 'default'
)


def formatar_molduras(ax):
  ax.spines['top'].set_visible(False)
  ax.spines['bottom'].set_color('black')
  ax.spines['left'].set_color('black')
  ax.spines['right'].set_color('black')


print('=' * 80)
print(
    '--- [BANCADA ECONOMÉTRICA]: TESTES DE CAUSALIDADE (VIX TUPINIQUIM II vs EPU'
    ' BRASIL) ---'
)
print('=' * 80)

# ==============================================================================
# 1. CARGA E ALINHAMENTO DAS SÉRIES HISTÓRICAS
# ==============================================================================
df_epu = pd.read_excel(
    'epu_baker_bloom_davis_brazil.xlsx', sheet_name='Brazil EPU Index'
)
df_epu['Data'] = pd.to_datetime(
    df_epu['year'].astype(str) + '-' + df_epu['month'].astype(str) + '-01'
)
df_epu = df_epu.rename(columns={'Brazil News-Based EPU': 'EPU'})

df_vix = pd.read_excel('serie_historica_vix_tupiniquim_ii.xlsx')
df_vix['Data'] = pd.to_datetime(df_vix['Data'])

# ==============================================================================
# 2. FILTRO ESTRUTURAL (STL) E MERGE
# ==============================================================================
print('\nAplicando Filtro Estrutural STL (period=13) no EPU Brasil...')
stl_epu = STL(df_epu['EPU'], period=13, robust=True).fit()
df_epu['EPU_SA'] = stl_epu.trend + stl_epu.resid

df_analise = (
    pd.merge(df_vix, df_epu[['Data', 'EPU_SA']], on='Data', how='inner')
    .sort_values('Data')
    .reset_index(drop=True)
)
print(
    f'[ALINHAMENTO]: Amostra comum final com {len(df_analise)} observações'
    ' mensais sincronizadas.'
)

# ==============================================================================
# 3. VERIFICAÇÃO DE ESTACIONARIEDADE (TESTES ADF)
# ==============================================================================
print('\n' + '=' * 80)
print('                  1. TESTES DE ESTACIONARIEDADE (ADF)')
print('=' * 80)

p_vix = adfuller(df_analise['VIX_Tupiniquim_XGBoost'])[1]
p_epu = adfuller(df_analise['EPU_SA'])[1]

print(
    '-> VIX Tupiniquim II (Nível)        | P-valor ADF: '
    f'{p_vix:.4f} -> Estacionário I(0)'
)
print(
    '-> EPU Dessazonalizado (Nível)      | P-valor ADF: '
    f'{p_epu:.4f} -> Estacionário I(0)'
)
print('=' * 80)

# ==============================================================================
# 4. TESTE DE CAUSALIDADE DE GRANGER EM NÍVEL (AMBAS SÃO I(0))
# ==============================================================================
max_lags = 3
print('\n' + '=' * 80)
print(
    f'        2. TESTES DE CAUSALIDADE DE GRANGER EM NÍVEL (Lags 1 a {max_lags})'
)
print('=' * 80)

# Sentido 1: VIX Tupiniquim -> EPU_SA (Nível)
print('\n[SENTIDO 1]: VIX Tupiniquim (Mercado) -> EPU (Notícias/Imprensa)')
gc_vix_to_epu = grangercausalitytests(
    df_analise[['EPU_SA', 'VIX_Tupiniquim_XGBoost']],
    maxlag=max_lags,
    verbose=False,
)
for lag in range(1, max_lags + 1):
  p_val = gc_vix_to_epu[lag][0]['ssr_ftest'][1]
  status = (
      'REJEITA H0 (Granger-Causa)'
      if p_val < 0.05
      else 'Não Rejeita H0 (Sem Causalidade)'
  )
  print(f'-> Lag {lag}: P-valor = {p_val:.4f} | {status}')

# Sentido 2: EPU_SA (Nível) -> VIX Tupiniquim
print('\n[SENTIDO 2]: EPU (Notícias/Imprensa) -> VIX Tupiniquim (Mercado)')
gc_epu_to_vix = grangercausalitytests(
    df_analise[['VIX_Tupiniquim_XGBoost', 'EPU_SA']],
    maxlag=max_lags,
    verbose=False,
)
for lag in range(1, max_lags + 1):
  p_val = gc_epu_to_vix[lag][0]['ssr_ftest'][1]
  status = (
      'REJEITA H0 (Granger-Causa)'
      if p_val < 0.05
      else 'Não Rejeita H0 (Sem Causalidade)'
  )
  print(f'-> Lag {lag}: P-valor = {p_val:.4f} | {status}')
print('=' * 80)

# ==============================================================================
# 5. PROCEDIMENTO DE TODA-YAMAMOTO (1995)
# ==============================================================================
print('\n' + '=' * 80)
print('            3. TESTE DE CAUSALIDADE DE TODA-YAMAMOTO')
print('=' * 80)

d_max = 1
var_model = VAR(df_analise[['VIX_Tupiniquim_XGBoost', 'EPU_SA']])
lag_order = var_model.select_order(maxlags=6)
k = max(lag_order.bic, 1)

print(f'-> Ordem ótima do VAR em nível (k): {k} lag(s)')
print(f'-> Ordem máxima de integração (d_max): {d_max}')
print(f'-> VAR aumentado estimado com (k + d_max) = {k + d_max} lags em nível.\n')


def executar_toda_yamamoto(df_data, y_nome, x_nome, k_lags, d_integracao):
  df_ty = pd.DataFrame(index=df_data.index)
  df_ty['const'] = 1.0

  for i in range(1, k_lags + d_integracao + 1):
    df_ty[f'{y_nome}_lag{i}'] = df_data[y_nome].shift(i)

  for i in range(1, k_lags + d_integracao + 1):
    df_ty[f'{x_nome}_lag{i}'] = df_data[x_nome].shift(i)

  df_ty['target'] = df_data[y_nome]
  df_reg = df_ty.dropna()

  X_mat = df_reg.drop(columns=['target'])
  y_vec = df_reg['target']

  modelo_ols = sm.OLS(y_vec, X_mat).fit(cov_type='HC1')

  restricoes = [f'{x_nome}_lag{i} = 0' for i in range(1, k_lags + 1)]
  formula_wald = ', '.join(restricoes)

  teste_wald = modelo_ols.wald_test(formula_wald, scalar=True)
  return teste_wald.statistic, teste_wald.pvalue


stat_1, p_val_1 = executar_toda_yamamoto(
    df_analise, 'EPU_SA', 'VIX_Tupiniquim_XGBoost', k, d_max
)
status_1 = (
    'REJEITA H0 (Causa)' if p_val_1 < 0.05 else 'Não Rejeita H0 (Sem Causalidade)'
)
print('[SENTIDO 1 (TY)]: VIX Tupiniquim -> EPU_SA (Nível)')
print(f'-> Estatística Wald: {stat_1:.4f} | P-valor = {p_val_1:.4f} | {status_1}')

stat_2, p_val_2 = executar_toda_yamamoto(
    df_analise, 'VIX_Tupiniquim_XGBoost', 'EPU_SA', k, d_max
)
status_2 = (
    'REJEITA H0 (Causa)' if p_val_2 < 0.05 else 'Não Rejeita H0 (Sem Causalidade)'
)
print('\n[SENTIDO 2 (TY)]: EPU_SA (Nível) -> VIX Tupiniquim')
print(f'-> Estatística Wald: {stat_2:.4f} | P-valor = {p_val_2:.4f} | {status_2}')
print('=' * 80)

# ==============================================================================
# 6. GRÁFICO COMPARATIVO: VIX TUPINIQUIM II vs. EPU BRASIL
# ==============================================================================
print(
    '\nGerando Gráfico Comparativo: VIX Tupiniquim II vs. EPU Brasil (Dois'
    ' Eixos)...'
)

fig, ax1 = plt.subplots(figsize=(14, 5.2))

# Eixo 1: VIX Tupiniquim II (Laranja Escuro)
ax1.plot(
    df_analise['Data'],
    df_analise['VIX_Tupiniquim_XGBoost'],
    color='darkorange',
    linewidth=2.2,
)
ax1.set_xlabel('Anos', fontweight='bold')
ax1.set_ylabel(
    'VIX Tupiniquim II (Pontos)', fontweight='bold', color='darkorange'
)
ax1.tick_params(axis='y', labelcolor='darkorange')
ax1.grid(True, linestyle=':', alpha=0.4)

# Eixo 2: EPU Dessazonalizado (Azul Tracejado)
ax2 = ax1.twinx()
ax2.plot(
    df_analise['Data'],
    df_analise['EPU_SA'],
    color='#1f77b4',
    linewidth=1.8,
    linestyle='--',
)
ax2.set_ylabel('EPU Index (Pontos)', fontweight='bold', color='#1f77b4')
ax2.tick_params(axis='y', labelcolor='#1f77b4')

formatar_molduras(ax1)
formatar_molduras(ax2)

plt.tight_layout()
plt.savefig('vix_vs_epu_pt.png', dpi=300, bbox_inches='tight')
plt.show()

print("[SUCESSO]: 'vix_vs_epu_pt.png' gerado com sucesso!")